In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from lightgbm import LGBMRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from joblib import dump
import time
import warnings
warnings.filterwarnings('ignore')

# Function to create a named pipeline
def create_pipeline(steps_list):
    """Helper function to create a named pipeline."""
    return Pipeline(steps_list)

# === Loading data ===
print("Loading data...")
data = pd.read_csv('egfr_small_molecules.csv', index_col=0)

# Check for target variable
if 'label' not in data.columns:
    raise ValueError("Target variable 'pIC50' not found in data")

# Separate features from target
X = data.drop(columns=['label', 'smiles'])
y = data['label']

# Check for missing values
if X.isnull().sum().any() or y.isnull().any():
    print("Warning: There are missing values in the data. Filling with medians...")
    X = X.fillna(X.median())
    y = y.fillna(y.median())

print(f"Data size: {X.shape}")
print(f"Target variable: min={y.min():.3f}, max={y.max():.3f}, mean={y.mean():.3f}")

# Split data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

# === Create a named pipeline ===
pipe = create_pipeline([
       ('lightgbm', LGBMRegressor(
        random_state=42,
        n_jobs=1,  # Use all available cores
        verbose=-1  # Disable output
    ))
])

# === Hyperparameter tuning ===
param_grid = {
    'lightgbm__num_leaves': [31, 63, 127],
    'lightgbm__learning_rate': [0.01, 0.05, 0.1],
    'lightgbm__n_estimators': [100, 200, 300],
    'lightgbm__max_depth': [6, 8, -1],  # -1 means no limit
    'lightgbm__subsample': [0.8, 1.0]  # To prevent overfitting
}

# === Training process with grid search ===
print("\nStarting grid search...")
start_time = time.time()

grid_search = GridSearchCV(
    pipe, 
    param_grid, 
    cv=3, 
    scoring='neg_mean_squared_error',
    n_jobs=-1,  # Parallel computing
    verbose=1  # Progress bar
)

grid_search.fit(X_train, y_train)

print(f"Search completed in {time.time() - start_time:.2f} seconds")

# Save the best model
dump(grid_search.best_estimator_, 'best_model.joblib')
print("Best model saved as 'best_model.joblib'")

# === Prediction and evaluation ===
best_model = grid_search.best_estimator_

# Predictions on training and test data
train_predictions = best_model.predict(X_train)
test_predictions = best_model.predict(X_test)

print("\n=== RESULTS ===")
print("\nTraining set:")
print(f"R² Score: {r2_score(y_train, train_predictions):.4f}")
print(f"MSE: {mean_squared_error(y_train, train_predictions):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_train, train_predictions)):.4f}")
print(f"MAE: {mean_absolute_error(y_train, train_predictions):.4f}")

print("\nTest set:")
print(f"R² Score: {r2_score(y_test, test_predictions):.4f}")
print(f"MSE: {mean_squared_error(y_test, test_predictions):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, test_predictions)):.4f}")
print(f"MAE: {mean_absolute_error(y_test, test_predictions):.4f}")

# Additionally, print the optimal hyperparameter combination
print("\nBest parameters:")
for key, value in grid_search.best_params_.items():
    print(f"{key}: {value}")

print(f"\nBest result (Negative MSE): {grid_search.best_score_:.4f}")

# Check for overfitting
train_test_gap = abs(r2_score(y_train, train_predictions) - r2_score(y_test, test_predictions))
print(f"\nR² difference between train and test: {train_test_gap:.4f}")
if train_test_gap > 0.1:
    print("Warning: Possible overfitting!")
else:
    print("No overfitting detected.")

In [ ]:
from pathlib import Path
def load_pipeline(model_path: str) -> Pipeline:
    """
    Loads a trained model pipeline from a file.

    Args:
        model_path (str | Path): Path to the model file (*.joblib).

    Returns:
        Pipeline: The loaded scikit-learn.
    
    Raises:
        FileNotFoundError: If the model file is not found.
        Exception: For other loading errors.
    """
    if not Path(model_path).exists():
        raise FileNotFoundError(f"Model file not found at {model_path}")
    try:
        model = joblib.load(model_path)
        print(f"✓ Model successfully loaded from {model_path}")
        return model
    except Exception as e:
        print(f"X Error occurred while loading the model: {e}")
        raise

In [ ]:
from joblib import load  

def load_pipeline(model_path='best_model.joblib'):
    """Function to load the saved model"""
    try:
        model = load(model_path)
        print(f"✓ Model successfully loaded from {model_path}")
        return model
    except FileNotFoundError:
        raise FileNotFoundError(f"Model file not found at {model_path}")
    except Exception as e:
        print(f"Error loading the model: {e}")
        return None

model = load_pipeline('best_model.joblib')

In [ ]:
import pandas as pd
from sklearn.pipeline import Pipeline

def predict(model: Pipeline, data2m2m: pd.DataFrame, training_columns: list[str], smiles_col: str) -> pd.DataFrame:
    """
    Performs pIC50 predictions for new data using a trained model.

    Parameters:
    model (Pipeline): Trained Pipeline model.
    data2m2m (pd.DataFrame): DataFrame with data for prediction.
    training_columns (list[str]): List of columns used during training.
    smiles_col (str): Name of the column with SMILES.

    Returns:
    pd.DataFrame: DataFrame with columns SMILES and predicted_pIC50.
    """
    if model is None:
        raise ValueError("Model not loaded. Load the model first.")
    
    # Check if all required columns are present in the data
    missing_cols = set(training_columns) - set(data2m2m.columns)
    if missing_cols:
        raise ValueError(f"Missing columns that were present during training: {missing_cols}")
    
    # Ensure the data has the same columns as during training, in the same order
    X_new = data2m2m[training_columns]
    
    # Make predictions
    predictions = model.predict(X_new)
    
    # Create a DataFrame with the results
    result_df = pd.DataFrame({
        smiles_col: data2m2m[smiles_col],
        'predicted_pIC50': predictions
    })
    
    return result_df

In [12]:
preds = predict(model,data,X_train.columns,"smiles")

In [13]:
preds.to_csv("pred_lgbm_m2m_similarity.csv")